In [ ]:
# ============================================================
# Interactive Multi-Class Traffic MFG
# 1. Imports and environment verification
# ============================================================

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output

# Locate the repository root.
cwd = Path.cwd().resolve()

if (cwd / "src" / "newton_krylov" / "multi_class").is_dir():
    ROOT = cwd
elif (cwd.parent / "src" / "newton_krylov" / "multi_class").is_dir():
    ROOT = cwd.parent
else:
    raise RuntimeError(
        "Launch JupyterLab from the repository root "
        "or its notebooks directory."
    )

SOLVER_DIR = ROOT / "src" / "newton_krylov" / "multi_class"

if str(SOLVER_DIR) not in sys.path:
    sys.path.insert(0, str(SOLVER_DIR))

from inputs import get_inputs
from main import run_solver

#print("=" * 60)
#print("INTERACTIVE MULTI-CLASS TRAFFIC MFG")
#print("=" * 60)

#print("Repository:", ROOT)
#print("NumPy:", np.__version__)
#print("ipywidgets:", widgets.__version__)

#print("\nNumerical solver imported successfully.")

In [ ]:
# ============================================================
# 2. Interactive traffic model configuration
# ============================================================

# Traffic model
model_widget = widgets.Dropdown(
    options=[
        ("Generalized LWR (GLWR)", "GLWR"),
        ("Generalized Separable (GS)", "GS"),
        ("Generalized Non-Separable (GNS)", "GNS"),
    ],
    value="GLWR",
    description="Model:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="450px"),
)

# Number of vehicle classes
classes_widget = widgets.BoundedIntText(
    value=2,
    min=1,
    max=10,
    step=1,
    description="Vehicle classes:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="450px"),
)

# Initial traffic configuration
configuration_widget = widgets.Dropdown(
    options=[
        ("Default class distributions", "DEFAULT"),
        ("Cars–Trucks (CT)", "CT"),
        ("Trucks–Cars (TC)", "TC"),
        ("Trucks–Cars–Trucks (TCT)", "TCT"),
    ],
    value="TC",
    description="Initial condition:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="450px"),
)

# Gaussian width
sigma_widget = widgets.BoundedFloatText(
    value=0.15,
    min=0.01,
    max=1.0,
    step=0.01,
    description="Gaussian width:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="450px"),
)

# Simulation time horizon
time_widget = widgets.BoundedFloatText(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description="Time horizon T:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="450px"),
)


# ============================================================
# 3. Configuration-dependent controls
# ============================================================

def update_configuration_options(change=None):
    """
    CT, TC, and TCT are currently implemented for J=2.
    Other class counts use the solver's default profiles.
    """

    if classes_widget.value == 2:
        configuration_widget.options = [
            ("Default class distributions", "DEFAULT"),
            ("Cars–Trucks (CT)", "CT"),
            ("Trucks–Cars (TC)", "TC"),
            ("Trucks–Cars–Trucks (TCT)", "TCT"),
        ]
    else:
        configuration_widget.options = [
            ("Default class distributions", "DEFAULT"),
        ]

    sigma_widget.disabled = (
        configuration_widget.value == "DEFAULT"
    )


classes_widget.observe(
    update_configuration_options,
    names="value",
)

configuration_widget.observe(
    update_configuration_options,
    names="value",
)

update_configuration_options()


# ============================================================
# 4. Display the configuration panel
# ============================================================

configuration_panel = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Traffic Model Configuration</h3>"
        ),
        model_widget,
        classes_widget,
        configuration_widget,
        sigma_widget,
        time_widget,
    ],
    layout=widgets.Layout(
        border="1px solid #d0d7de",
        padding="20px",
        width="520px",
    ),
)

#display(configuration_panel)

In [ ]:
# ============================================================
# 5. Interactive numerical solver configuration
# ============================================================

# Spatial resolution
nx_widget = widgets.BoundedIntText(
    value=5,
    min=3,
    max=200,
    step=1,
    description="Spatial cells Nx:",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="470px"),
)

# Temporal resolution
nt_widget = widgets.BoundedIntText(
    value=8,
    min=2,
    max=1000,
    step=1,
    description="Time steps Nt:",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="470px"),
)

# Grid refinement
grids_widget = widgets.BoundedIntText(
    value=1,
    min=1,
    max=5,
    step=1,
    description="Grid levels:",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="470px"),
)

# Newton tolerance
newton_tol_widget = widgets.Dropdown(
    options=[
        ("1e-6", 1e-6),
        ("1e-8", 1e-8),
        ("1e-10", 1e-10),
        ("1e-12", 1e-12),
    ],
    value=1e-8,
    description="Newton tolerance:",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="470px"),
)

# GMRES tolerance
gmres_tol_widget = widgets.Dropdown(
    options=[
        ("1e-6", 1e-6),
        ("1e-8", 1e-8),
        ("1e-10", 1e-10),
        ("1e-12", 1e-12),
    ],
    value=1e-8,
    description="GMRES tolerance:",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="470px"),
)

# Maximum Newton iterations
max_newton_widget = widgets.BoundedIntText(
    value=50,
    min=1,
    max=200,
    step=1,
    description="Maximum Newton iterations:",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="470px"),
)

# Preconditioning
precond_widget = widgets.Checkbox(
    value=True,
    description="Enable sparse LU preconditioning",
    indent=False,
)

# Multigrid interpolation
multigrid_widget = widgets.Checkbox(
    value=True,
    description="Enable interpolation between refined grids",
    indent=False,
)


# ============================================================
# 6. Display numerical configuration panel
# ============================================================

numerical_panel = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Numerical Solver Configuration</h3>"
        ),
        nx_widget,
        nt_widget,
        grids_widget,
        newton_tol_widget,
        gmres_tol_widget,
        max_newton_widget,
        widgets.HTML("<hr>"),
        precond_widget,
        multigrid_widget,
    ],
    layout=widgets.Layout(
        border="1px solid #d0d7de",
        padding="20px",
        width="520px",
    ),
)

#display(numerical_panel)

In [ ]:
# ============================================================
# 7. Verify the initial-condition interface
# ============================================================

import inspect

DEMO_DIR = ROOT / "examples" / "multi_class_demo"

if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

from initial_conditions import (
    get_initial_configuration,
    validate_initial_configuration,
)

#print("=" * 60)
#print("INITIAL-CONDITION INTERFACE")
#print("=" * 60)

#print(
#    "get_initial_configuration",
#    inspect.signature(get_initial_configuration),
#)

#print(
#    "validate_initial_configuration",
#    inspect.signature(validate_initial_configuration),
#)

#print("\nTC configuration:")

#parameters = get_initial_configuration("TC")

#for key, value in parameters.items():
#    if key == "rho_int_list":
#        print(f"{key}: {len(value)} density functions")
#    else:
#        print(f"{key}: {value}")

In [ ]:
# ============================================================
# 8. Interactive simulation execution
# ============================================================

from argparse import Namespace
import time
import traceback

# Store the latest simulation results.
simulation_state = {
    "results": None,
    "parameters": None,
    "elapsed": None,
}

# Run button
run_button = widgets.Button(
    description="Run Simulation",
    button_style="success",
    icon="play",
    layout=widgets.Layout(
        width="200px",
        height="42px",
    ),
)

# Output area for solver information
simulation_output = widgets.Output()


def run_interactive_simulation(button):
    """
    Execute the sequential Newton-GMRES solver using
    the parameters selected in the interactive interface.
    """

    run_button.disabled = True

    # Prevent previously computed results from being mistaken
    # for the results of a new simulation.
    simulation_state["results"] = None
    simulation_state["parameters"] = None
    simulation_state["elapsed"] = None

    try:
        with simulation_output:
            clear_output(wait=True)

            # --------------------------------------------
            # 1. Read model configuration
            # --------------------------------------------

            J = classes_widget.value
            model = model_widget.value
            configuration = configuration_widget.value
            sigma = sigma_widget.value
            T = time_widget.value

            # --------------------------------------------
            # 2. Load physical parameters
            # --------------------------------------------

            parameters = get_inputs(
                J=J,
                model=model,
            )

            parameters["T"] = T

            # --------------------------------------------
            # 3. Apply selected initial conditions
            # --------------------------------------------

            if configuration != "DEFAULT":

                if J != 2:
                    raise ValueError(
                        "CT, TC, and TCT require J=2."
                    )

                initial_parameters = get_initial_configuration(
                    configuration=configuration,
                    sigma=sigma,
                )

                validate_initial_configuration(
                    initial_parameters
                )

                parameters["L"] = initial_parameters["L"]

                parameters["rho_int_list"] = (
                    initial_parameters["rho_int_list"]
                )

            # --------------------------------------------
            # 4. Read numerical solver parameters
            # --------------------------------------------

            args = Namespace(
                J=J,
                model=model,
                nx=nx_widget.value,
                nt=nt_widget.value,
                max_grids=grids_widget.value,
                no_precond=not precond_widget.value,
                no_multigrid=not multigrid_widget.value,
                newton_tol=newton_tol_widget.value,
                gmres_rtol=gmres_tol_widget.value,
                max_newton_iter=max_newton_widget.value,
                output=None,
            )

            # --------------------------------------------
            # 5. Display selected configuration
            # --------------------------------------------

            print("=" * 65)
            print("INTERACTIVE MULTI-CLASS TRAFFIC MFG SIMULATION")
            print("=" * 65)

            print(f"Model:               {model}")
            print(f"Vehicle classes:     {J}")
            print(f"Initial condition:   {configuration}")
            print(f"Time horizon:        {T}")
            print(f"Road length:         {parameters['L']}")
            print(f"Spatial cells:       {args.nx}")
            print(f"Minimum time steps:  {args.nt}")
            print(f"Grid levels:         {args.max_grids}")

            print("=" * 65)

            # --------------------------------------------
            # 6. Execute numerical solver
            # --------------------------------------------

            start = time.perf_counter()

            results = run_solver(
                args,
                parameters=parameters,
            )

            elapsed = time.perf_counter() - start

            # --------------------------------------------
            # 7. Store results
            # --------------------------------------------

            simulation_state["results"] = results
            simulation_state["parameters"] = parameters
            simulation_state["elapsed"] = elapsed

            # --------------------------------------------
            # 8. Display numerical summary
            # --------------------------------------------

            print("\n" + "=" * 65)
            print("INTERACTIVE SIMULATION FINISHED")
            print("=" * 65)

            print(
                "Density array:",
                results["rho"].shape,
            )

            print(
                "Velocity array:",
                results["u"].shape,
            )

            print(
                "Value array:",
                results["V"].shape,
            )

            print(
                "Final residual:",
                f"{results['residual_l2']:.6e}",
            )

            print(
                "Execution time:",
                f"{elapsed:.3f} seconds",
            )

    except Exception:
        with simulation_output:
            print("\nSimulation failed:")
            traceback.print_exc()

    finally:
        run_button.disabled = False


# Connect the button to the solver.
run_button.on_click(run_interactive_simulation)

# Display simulation controls.
#display(
#    widgets.VBox(
#        [
#            widgets.HTML(
#                "<h3>Execute Numerical Simulation</h3>"
#            ),
#            run_button,
#            simulation_output,
#        ]
#    )
#)

In [ ]:
# ============================================================
# 9. Interactive solution visualization
# ============================================================

variable_widget = widgets.Dropdown(
    options=[
        ("Traffic density", "rho"),
        ("Optimal velocity", "u"),
        ("Value function", "V"),
        ("Traffic flux", "Q"),
    ],
    value="rho",
    description="Variable:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"),
)

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=8,
    step=1,
    description="Time step:",
    continuous_update=False,
    style={"description_width": "100px"},
    layout=widgets.Layout(width="550px"),
)

plot_output = widgets.Output()


def update_solution_plot(change=None):
    """
    Plot the selected numerical variable at the
    selected simulation time.
    """

    with plot_output:
        clear_output(wait=True)

        results = simulation_state["results"]
        parameters = simulation_state["parameters"]

        if results is None:
            print("Run a simulation first.")
            return

        variable = variable_widget.value

        data = results[variable]

        J, Nx, n_times = data.shape

        # Extract the spatial grid.
        x_edges = results["x"]

        x_centers = 0.5 * (
            x_edges[:-1] + x_edges[1:]
        )

        # Extract the temporal grid.
        t = results["t"]

        # Density and value functions have Nt+1 time levels.
        # Velocity and flux have Nt time levels.
        time_slider.max = n_times - 1

        n = min(time_slider.value, n_times - 1)

        # Physical time corresponding to the selected index.
        current_time = t[n]

        # Variable labels.
        labels = {
            "rho": r"Traffic density $\rho_j$",
            "u": r"Optimal velocity $u_j$",
            "V": r"Value function $V_j$",
            "Q": r"Traffic flux $Q_j$",
        }

        # Vehicle-class labels.
        if J == 2:
            class_names = ["Cars", "Trucks"]
        else:
            class_names = [
                f"Class {j + 1}"
                for j in range(J)
            ]

        # --------------------------------------------
        # Create scientific figure
        # --------------------------------------------

        fig, ax = plt.subplots(
            figsize=(9, 5)
        )

        for j in range(J):

            ax.plot(
                x_centers,
                data[j, :, n],
                marker="o",
                linewidth=2,
                markersize=5,
                label=class_names[j],
            )

        ax.set_xlabel(
            r"Road position $x$",
            fontsize=12,
        )

        ax.set_ylabel(
            labels[variable],
            fontsize=12,
        )

        ax.set_title(
            f"{parameters['model']} — "
            f"{labels[variable]} at "
            f"t = {current_time:.3f}",
            fontsize=14,
        )

        ax.set_xlim(
            x_edges[0],
            x_edges[-1],
        )

        ax.grid(
            True,
            linestyle="--",
            alpha=0.3,
        )

        ax.legend()

        fig.tight_layout()

        plt.show()
        plt.close(fig)


# Connect widgets to plotting function.
variable_widget.observe(
    update_solution_plot,
    names="value",
)

time_slider.observe(
    update_solution_plot,
    names="value",
)


# Display visualization interface.
visualization_panel = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Interactive Numerical Visualization</h3>"
        ),
        variable_widget,
        time_slider,
        plot_output,
    ]
)

#display(visualization_panel)

# Draw the initial figure.
update_solution_plot()

In [ ]:
# ============================================================
# 10. NetLogo-inspired dashboard layout
# ============================================================

# Compact sidebar containing simulation controls.
control_sidebar = widgets.VBox(
    [
        widgets.HTML(
            "<h2 style='margin:0 0 12px 0;'>Simulation Controls</h2>"
        ),
        configuration_panel,
        numerical_panel,
        run_button,
    ],
    layout=widgets.Layout(
        width="550px",
        min_width="550px",
        padding="12px",
        border="1px solid #d0d7de",
        align_items="center",
    ),
)

# Main area containing the scientific figures.
results_area = widgets.VBox(
    [
        widgets.HTML(
            "<h2 style='margin:0 0 12px 0;'>Simulation Results</h2>"
        ),
        variable_widget,
        time_slider,
        plot_output,
    ],
    layout=widgets.Layout(
        flex="1 1 auto",
        min_width="500px",
        padding="20px",
        border="1px solid #d0d7de",
    ),
)

# Main application layout.
dashboard = widgets.HBox(
    [
        control_sidebar,
        results_area,
    ],
    layout=widgets.Layout(
        width="100%",
        align_items="flex-start",
        gap="20px",
    ),
)


In [ ]:
# ============================================================
# 11. Simulation status and automatic visualization refresh
# ============================================================

# Compact simulation status.
status_widget = widgets.HTML(
    value="""
    <div style="
        padding:12px;
        background:#f6f8fa;
        border-radius:6px;
        color:#57606a;
    ">
        Ready to run a simulation.
    </div>
    """
)

# Optional detailed solver output.
# Hidden by default, but available for debugging.
diagnostics_panel = widgets.Accordion(
    children=[simulation_output],
    selected_index=None,
)

diagnostics_panel.set_title(
    0,
    "Solver diagnostics"
)


def refresh_dashboard_after_simulation(button):
    """
    Update the scientific figures and simulation status
    after the Newton-GMRES solver has finished.
    """

    results = simulation_state["results"]

    # --------------------------------------------------------
    # Handle failed simulations
    # --------------------------------------------------------

    if results is None:

        status_widget.value = """
        <div style="
            padding:12px;
            background:#ffebe9;
            border:1px solid #cf222e;
            border-radius:6px;
            color:#82071e;
        ">
            <b>Simulation failed.</b>
            Open Solver diagnostics for details.
        </div>
        """

        with plot_output:
            clear_output(wait=True)
            print("No valid simulation results.")

        return

    # --------------------------------------------------------
    # Extract convergence information
    # --------------------------------------------------------

    residual = float(results["residual_l2"])
    elapsed = float(simulation_state["elapsed"])

    # Determine convergence against the requested tolerance.
    converged = (
        np.isfinite(residual)
        and residual <= newton_tol_widget.value
    )

    if converged:

        status_color = "#dafbe1"
        border_color = "#1a7f37"
        text_color = "#116329"
        status_text = "Simulation converged"

    else:

        status_color = "#fff8c5"
        border_color = "#9a6700"
        text_color = "#7d4e00"
        status_text = "Residual exceeds requested tolerance"

    # --------------------------------------------------------
    # Display compact numerical summary
    # --------------------------------------------------------

    status_widget.value = f"""
    <div style="
        padding:14px;
        background:{status_color};
        border:1px solid {border_color};
        border-radius:6px;
        color:{text_color};
        line-height:1.8;
    ">
        <b>{status_text}</b><br>
        Final residual: {residual:.3e}<br>
        Computation time: {elapsed:.3f} s
    </div>
    """

    # --------------------------------------------------------
    # Refresh the scientific visualization
    # --------------------------------------------------------

    time_slider.value = 0

    update_solution_plot()


# Register the refresh function after the solver callback.
run_button.on_click(
    refresh_dashboard_after_simulation
)


# ------------------------------------------------------------
# Add status information to the dashboard
# ------------------------------------------------------------

results_area.children = (
    results_area.children[0],
    status_widget,
    results_area.children[1],
    results_area.children[2],
    results_area.children[3],
    diagnostics_panel,
)

In [ ]:
# ============================================================
# 12. Scientific visualization tabs
# ============================================================

# Existing interactive spatial profiles.
spatial_profiles_tab = widgets.VBox(
    [
        variable_widget,
        time_slider,
        plot_output,
    ],
    layout=widgets.Layout(
        padding="15px",
    ),
)

# Additional scientific visualization areas.
spacetime_output = widgets.Output()
snapshots_output = widgets.Output()
mass_output = widgets.Output()
initial_output = widgets.Output()

# Create tabs.
results_tabs = widgets.Tab(
    children=[
        spatial_profiles_tab,
        spacetime_output,
        snapshots_output,
        mass_output,
        initial_output,
    ],
    layout=widgets.Layout(
        width="100%",
    ),
)

# Assign tab names.
tab_names = [
    "Spatial Profiles",
    "Space-Time",
    "Density Snapshots",
    "Mass Conservation",
    "Initial Conditions",
]

for index, name in enumerate(tab_names):
    results_tabs.set_title(index, name)



# ============================================================
# 13. Update the dashboard
# ============================================================

results_area.children = (
    results_area.children[0],  # Simulation Results heading
    status_widget,
    results_tabs,
    diagnostics_panel,
)

# ============================================================
# 14. Interactive space-time visualization
# ============================================================

spacetime_variable_widget = widgets.Dropdown(
    options=[
        ("Traffic density", "rho"),
        ("Optimal velocity", "u"),
        ("Traffic flux", "Q"),
        ("Value function", "V"),
    ],
    value="rho",
    description="Variable:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"),
)


def update_spacetime_plot(change=None):
    """
    Display the space-time evolution of the selected
    numerical variable for every vehicle class.
    """

    with spacetime_output:
        clear_output(wait=True)

        results = simulation_state["results"]

        if results is None:
            print("Run a simulation first.")
            return

        variable = spacetime_variable_widget.value
        data = results[variable]

        J, Nx, _ = data.shape

        x_edges = np.asarray(results["x"])
        t = np.asarray(results["t"])

        labels = {
            "rho": r"Traffic density $\rho_j$",
            "u": r"Optimal velocity $u_j$",
            "Q": r"Traffic flux $Q_j$",
            "V": r"Value function $V_j$",
        }

        if J == 2:
            class_names = ["Cars", "Trucks"]
        else:
            class_names = [
                f"Class {j + 1}"
                for j in range(J)
            ]

        # --------------------------------------------
        # Create class-wise heatmaps
        # --------------------------------------------

        fig, axes = plt.subplots(
            J,
            1,
            figsize=(10, 3.3 * J),
            squeeze=False,
            constrained_layout=True,
        )

        # Use the same color scale for every class.
        vmin = float(np.min(data))
        vmax = float(np.max(data))

        if np.isclose(vmin, vmax):
            # Constant solutions, such as V=0 in GLWR.
            delta = max(abs(vmin) * 0.05, 1e-6)
            vmin -= delta
            vmax += delta

        for j in range(J):

            ax = axes[j, 0]

            image = ax.imshow(
                data[j],
                origin="lower",
                aspect="auto",
                interpolation="nearest",
                extent=[
                    t[0],
                    t[-1],
                    x_edges[0],
                    x_edges[-1],
                ],
                cmap="viridis",
                vmin=vmin,
                vmax=vmax,
            )

            ax.set_title(
                class_names[j],
                fontsize=13,
            )

            ax.set_ylabel(
                r"Road position $x$",
                fontsize=11,
            )

            ax.set_xlabel(
                r"Time $t$",
                fontsize=11,
            )

            fig.colorbar(
                image,
                ax=ax,
                label=labels[variable],
            )

        fig.suptitle(
            f"{simulation_state['parameters']['model']}"
            f" — {labels[variable]} evolution",
            fontsize=15,
        )

        plt.show()
        plt.close(fig)


# Update the heatmap when the user changes the variable.
spacetime_variable_widget.observe(
    update_spacetime_plot,
    names="value",
)

# Update the heatmap after each simulation.
run_button.on_click(
    update_spacetime_plot
)


# ============================================================
# 15. Configure the Space-Time tab
# ============================================================

spacetime_tab = widgets.VBox(
    [
        spacetime_variable_widget,
        spacetime_output,
    ],
    layout=widgets.Layout(
        padding="15px",
    ),
)

current_tabs = list(results_tabs.children)

current_tabs[1] = spacetime_tab

results_tabs.children = tuple(current_tabs)

# Restore tab titles.
for index, name in enumerate(tab_names):
    results_tabs.set_title(index, name)


# ============================================================
# 16. Density snapshots
# ============================================================

def update_density_snapshots(change=None):
    """
    Compare vehicle-class density profiles at
    the beginning, middle, and end of the simulation.
    """

    with snapshots_output:
        clear_output(wait=True)

        results = simulation_state["results"]

        if results is None:
            print("Run a simulation first.")
            return

        # --------------------------------------------
        # Extract numerical results
        # --------------------------------------------

        rho = results["rho"]
        x_edges = np.asarray(results["x"])
        t = np.asarray(results["t"])

        J, Nx, Nt_plus_one = rho.shape

        Nt = Nt_plus_one - 1

        x_centers = 0.5 * (
            x_edges[:-1] + x_edges[1:]
        )

        # --------------------------------------------
        # Select time snapshots
        # --------------------------------------------

        snapshot_indices = [
            0,
            Nt // 2,
            Nt,
        ]

        if J == 2:
            class_names = ["Cars", "Trucks"]
        else:
            class_names = [
                f"Class {j + 1}"
                for j in range(J)
            ]

        # --------------------------------------------
        # Create scientific figure
        # --------------------------------------------

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(14, 4.5),
            sharey=True,
            constrained_layout=True,
        )

        # Use a common vertical scale for comparison.
        max_density = float(np.max(rho))
        upper_limit = max(1.05 * max_density, 1e-6)

        for k, n in enumerate(snapshot_indices):

            ax = axes[k]

            for j in range(J):

                ax.plot(
                    x_centers,
                    rho[j, :, n],
                    marker="o",
                    markersize=4,
                    linewidth=2,
                    label=class_names[j],
                )

            ax.set_title(
                f"t = {t[n]:.3f}",
                fontsize=13,
            )

            ax.set_xlabel(
                r"Road position $x$",
                fontsize=11,
            )

            ax.set_xlim(
                x_edges[0],
                x_edges[-1],
            )

            ax.set_ylim(
                0,
                upper_limit,
            )

            ax.grid(
                True,
                linestyle="--",
                alpha=0.3,
            )

        axes[0].set_ylabel(
            r"Traffic density $\rho_j$",
            fontsize=11,
        )

        axes[-1].legend(
            loc="best"
        )

        model = simulation_state["parameters"]["model"]

        fig.suptitle(
            f"{model} — Traffic Density Evolution",
            fontsize=15,
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 17. Update snapshots after each simulation
# ============================================================

run_button.on_click(
    update_density_snapshots
)

# Draw snapshots if a simulation is already available.
update_density_snapshots()


# ============================================================
# 18. Class-wise mass conservation
# ============================================================

def update_mass_conservation(change=None):
    """
    Evaluate and visualize mass conservation for
    every vehicle class.
    """

    with mass_output:
        clear_output(wait=True)

        results = simulation_state["results"]

        if results is None:
            print("Run a simulation first.")
            return

        # --------------------------------------------
        # Extract numerical data
        # --------------------------------------------

        rho = np.asarray(results["rho"])
        t = np.asarray(results["t"])
        x_edges = np.asarray(results["x"])

        J, Nx, Nt_plus_one = rho.shape

        dx = float(x_edges[1] - x_edges[0])

        # --------------------------------------------
        # Compute class-wise total masses
        # --------------------------------------------

        # Shape: (J, Nt+1)
        mass = dx * np.sum(rho, axis=1)

        initial_mass = mass[:, 0]

        mass_error = (
            mass - initial_mass[:, np.newaxis]
        )

        max_errors = np.max(
            np.abs(mass_error),
            axis=1,
        )

        if J == 2:
            class_names = ["Cars", "Trucks"]
        else:
            class_names = [
                f"Class {j + 1}"
                for j in range(J)
            ]

        # --------------------------------------------
        # Create scientific figure
        # --------------------------------------------

        fig, ax = plt.subplots(
            figsize=(9, 4.5)
        )

        for j in range(J):

            ax.plot(
                t,
                mass[j],
                marker="o",
                markersize=4,
                linewidth=2,
                label=class_names[j],
            )

        ax.set_xlabel(
            r"Time $t$",
            fontsize=12,
        )

        ax.set_ylabel(
            r"Total mass $M_j(t)$",
            fontsize=12,
        )

        ax.set_title(
            "Class-Wise Mass Conservation",
            fontsize=14,
        )

        ax.grid(
            True,
            linestyle="--",
            alpha=0.3,
        )

        ax.legend()

        fig.tight_layout()

        plt.show()
        plt.close(fig)

        # --------------------------------------------
        # Display conservation diagnostics
        # --------------------------------------------

        print("=" * 55)
        print("MASS CONSERVATION DIAGNOSTICS")
        print("=" * 55)

        for j in range(J):

            print(
                f"{class_names[j]}:"
            )

            print(
                f"  Initial mass:  "
                f"{initial_mass[j]:.10e}"
            )

            print(
                f"  Final mass:    "
                f"{mass[j, -1]:.10e}"
            )

            print(
                f"  Maximum error: "
                f"{max_errors[j]:.3e}"
            )

            print()


# ============================================================
# 19. Refresh mass conservation after each simulation
# ============================================================

run_button.on_click(
    update_mass_conservation
)

# Draw the figure if results are already available.
update_mass_conservation()


# ============================================================
# 20. Interactive initial-condition visualization
# ============================================================

def update_initial_conditions(change=None):
    """
    Preview the initial traffic density distributions
    using the currently selected interface parameters.
    """

    with initial_output:
        clear_output(wait=True)

        try:
            # --------------------------------------------
            # Read selected parameters
            # --------------------------------------------

            J = classes_widget.value
            model = model_widget.value
            configuration = configuration_widget.value
            sigma = sigma_widget.value

            parameters = get_inputs(
                J=J,
                model=model,
            )

            # --------------------------------------------
            # Apply selected initial configuration
            # --------------------------------------------

            if configuration != "DEFAULT":

                if J != 2:
                    raise ValueError(
                        "CT, TC, and TCT require J=2."
                    )

                initial_parameters = get_initial_configuration(
                    configuration=configuration,
                    sigma=sigma,
                )

                validate_initial_configuration(
                    initial_parameters
                )

                parameters["L"] = initial_parameters["L"]

                parameters["rho_int_list"] = (
                    initial_parameters["rho_int_list"]
                )

            # --------------------------------------------
            # Extract physical data
            # --------------------------------------------

            L = float(parameters["L"])

            rho_int_list = parameters["rho_int_list"]

            if J == 2:
                class_names = ["Cars", "Trucks"]
            else:
                class_names = [
                    f"Class {j + 1}"
                    for j in range(J)
                ]

            # Smooth spatial grid for visualization.
            x_plot = np.linspace(
                0,
                L,
                600,
            )

            # --------------------------------------------
            # Plot prescribed initial densities
            # --------------------------------------------

            fig, ax = plt.subplots(
                figsize=(9, 4.5)
            )

            for j in range(J):

                rho_initial = rho_int_list[j](x_plot)

                ax.plot(
                    x_plot,
                    rho_initial,
                    linewidth=2.5,
                    label=class_names[j],
                )

                ax.fill_between(
                    x_plot,
                    rho_initial,
                    alpha=0.12,
                )

            ax.set_xlabel(
                r"Road position $x$",
                fontsize=12,
            )

            ax.set_ylabel(
                r"Initial density $\rho_j(0,x)$",
                fontsize=12,
            )

            ax.set_title(
                f"Initial Traffic Configuration — {configuration}",
                fontsize=14,
            )

            ax.set_xlim(0, L)
            ax.set_ylim(bottom=0)

            ax.grid(
                True,
                linestyle="--",
                alpha=0.3,
            )

            ax.legend()

            fig.tight_layout()

            plt.show()
            plt.close(fig)

        except Exception as exc:
            print(
                "Unable to display the initial configuration:"
            )
            print(exc)


# ============================================================
# 21. Connect initial-condition controls
# ============================================================

classes_widget.observe(
    update_initial_conditions,
    names="value",
)

model_widget.observe(
    update_initial_conditions,
    names="value",
)

configuration_widget.observe(
    update_initial_conditions,
    names="value",
)

sigma_widget.observe(
    update_initial_conditions,
    names="value",
)

# Display the initial profiles immediately.
update_initial_conditions()


# ============================================================
# 22. Detect changes to simulation parameters
# ============================================================

# Track the parameters used for the most recent simulation.
last_run_state = {
    "settings": None,
    "status_html": None,
}


def get_current_settings():
    """
    Return a snapshot of all settings that affect
    the numerical simulation.
    """

    return (
        model_widget.value,
        classes_widget.value,
        configuration_widget.value,
        sigma_widget.value,
        time_widget.value,
        nx_widget.value,
        nt_widget.value,
        grids_widget.value,
        newton_tol_widget.value,
        gmres_tol_widget.value,
        max_newton_widget.value,
        precond_widget.value,
        multigrid_widget.value,
    )


def record_simulation_settings(button):
    """
    Store the settings after a successful simulation.
    """

    if simulation_state["results"] is not None:

        last_run_state["settings"] = (
            get_current_settings()
        )

        last_run_state["status_html"] = (
            status_widget.value
        )


def check_parameter_changes(change=None):
    """
    Warn users if the selected settings differ from
    those used to compute the displayed results.
    """

    previous_settings = last_run_state["settings"]

    if previous_settings is None:
        return

    current_settings = get_current_settings()

    if current_settings != previous_settings:

        status_widget.value = """
        <div style="
            padding:14px;
            background:#fff8c5;
            border:1px solid #9a6700;
            border-radius:6px;
            color:#7d4e00;
            line-height:1.7;
        ">
            <b>Parameters changed</b><br>
            Run Simulation to update the numerical results.
        </div>
        """

    else:

        status_widget.value = (
            last_run_state["status_html"]
        )


# ============================================================
# 23. Connect parameter-change detection
# ============================================================

simulation_widgets = [
    model_widget,
    classes_widget,
    configuration_widget,
    sigma_widget,
    time_widget,
    nx_widget,
    nt_widget,
    grids_widget,
    newton_tol_widget,
    gmres_tol_widget,
    max_newton_widget,
    precond_widget,
    multigrid_widget,
]

for widget in simulation_widgets:

    widget.observe(
        check_parameter_changes,
        names="value",
    )


# Record settings after the solver finishes.
run_button.on_click(
    record_simulation_settings
)

# ============================================================
# Display the completed interactive dashboard
# ============================================================

display(
    widgets.VBox(
        [
            widgets.HTML(
                """
                <h1 style="margin-bottom:5px;">
                    Multi-Class Traffic MFG Simulator
                </h1>
                <p style="color:#57606a;">
                    Interactive numerical exploration of
                    GLWR, GS, and GNS traffic models
                </p>
                """
            ),
            dashboard,
        ],
        layout=widgets.Layout(width="100%"),
    )
)